In [5]:
import torch

# The following is vibe-coded with Gemini

# Use float64 for maximum precision with exponential terms
torch.set_default_dtype(torch.float64)

def calculate_J_with_viable_lambdas(R1, R2, S0, theta):
    """
    Computes J and its gradients while dynamically solving for
    arbitrage-free, viable lambdas at the x_strike junction.

    Layout of theta: [nu1 (R1), sig1_raw (R1), nu2 (R2), sig2_raw (R2)]
    No lambdas are inside theta anymore; they are solved analytically!

    NOTE TO SELF: sigs2 are flipped than what is outputted in Hwk_prob5b_sol
    """
    # 1. Slicing the parameters
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1_raw = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2_raw = theta[idx : idx + R2]; idx += R2

    nus1 = torch.cat([nus1, torch.tensor([S0])])
    nus2 = torch.cat([torch.tensor([S0]), nus2])

    # Enforce strict positivity on sigmas using softplus
    sigs1 = torch.nn.functional.softplus(sigs1_raw) + 1e-6
    sigs2 = torch.nn.functional.softplus(sigs2_raw) + 1e-6

    # =========================================================================
    # STAGE 1: Calculate Unit Base Coefficients (Setting lam1 = 1, lam2 = 1)
    # =========================================================================

    # Left Wing Unit Base
    c_v1_unit = []
    dist0 = nus1[1] - nus1[0]
    s0 = sigs1[0]
    # Boundary at L with lam1_unit = 1.0
    c_v1_unit.append((torch.exp(-dist0/s0), torch.exp(dist0/s0)))

    for j in range(R1 - 1): # check this range
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1.0/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1[j+2] - nus1[j+1]
        sn = sigs1[j+1]
        c_v1_unit.append((0.5 * (P - sn*S) * torch.exp(-dist/sn),
                          0.5 * (P + sn*S) * torch.exp(dist/sn)))

    # Right Wing Unit Base
    c_v2_unit = [None] * R2
    dist_u = nus2[-1] - nus2[-2]
    sn_u = sigs2[-1]
    # Boundary at U with lam2_unit = 1.0
    c_v2_unit[-1] = ((torch.exp(dist_u/sn_u), torch.exp(-dist_u/sn_u)))

    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1.0/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2[j-1] - nus2[j-2]
        sp = sigs2[j-1]
        c_v2_unit[j-1] = (0.5 * (P + sp*S) * torch.exp(dist/sp),
                          0.5 * (P - sp*S) * torch.exp(-dist/sp)) # check this j-1 thing

    # =========================================================================
    # STAGE 2: Evaluate Unit Wings at x_strike and Solve for Viable Lambdas
    # =========================================================================
    
    # Left wing unit value at x_strike
    v1 = c_v1_unit[-1][0] + c_v1_unit[-1][0]

    # Right wing unit value at x_strike
    v2 = c_v2_unit[0][0] + c_v2_unit[0][1]

    Dk_v1 = (1.0/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][0])

    Dk_v2 = (1.0/sigs2[0]) * (-c_v2_unit[0][0] + c_v2_unit[0][1])

    # Define viable lambdas
    lam1 = v2 / (Dk_v1 * v2 - v1 * Dk_v2)

    lam2 = v1 / (Dk_v1 * v2 - v1 * Dk_v2)

    # =========================================================================
    # STAGE 3: Scale Unit Coefficients to Final Viable Coefficients
    # =========================================================================
    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    # =========================================================================
    # STAGE 4: Calculate Smoothness Jumps Using Viable Slopes
    # =========================================================================
    jumps = []

    # Internal Left Wing Jumps
    for j in range(R1 - 1):
        v_sec_left = (1.0/sigs1[j]**2) * (c_v1[j][0] + c_v1[j][1])
        dist = nus1[j+2] - nus1[j+1]
        v_sec_right = (1.0/sigs1[j+1]**2) * (c_v1[j+1][0]*torch.exp(dist/sigs1[j+1]) +
                                            c_v1[j+1][1]*torch.exp(-dist/sigs1[j+1]))
        jumps.append(v_sec_right - v_sec_left)

    # Junction Jump at x_strike
    v_sec_v1_strike = (1.0/sigs1[-1]**2) * (c_v1[-1][0] + c_v1[-1][0])
    v_sec_v2_strike = (1.0/sigs2[0]**2) * (c_v2[0][0] + c_v2[0][1])
    
    jumps.append(v_sec_v2_strike - v_sec_v1_strike)

    # Internal Right Wing Jumps
    for j in range(R2 - 1):
        v_sec_right = (1.0/sigs2[j+1]**2) * (c_v2[j+1][0] + c_v2[j+1][1])
        dist = nus2[j+1] - nus2[j]
        v_sec_left = (1.0/sigs2[j]**2) * (c_v2[j][0]*torch.exp(-dist/sigs2[j+1]) +
                                            c_v2[j][1]*torch.exp(dist/sigs2[j+1]))
        jumps.append(v_sec_right - v_sec_left)

    j_val = torch.sum(torch.stack(jumps) ** 2)
    return j_val, lam1, lam2

# --- Verification execution ---
if __name__ == "__main__":
    R1, R2 = 4, 3

    # Financial Market Conditions
    S0 = 1271.87     # Spot asset price

    # Initialize unconstrained parameters
    # Sigmas initialized uniquely to [149, 130, 250, 160] and [155, 180, 210]
    initial_sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
    initial_sigs2 = torch.tensor([155.0, 180.0, 210.0])
    sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1.0)
    sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1.0)

    theta = torch.cat([
        torch.linspace(900, 1200, R1),   # nus1
        sigs1_raw,                       # sigs1_raw
        torch.linspace(1400, 2000, R2),  # nus2
        sigs2_raw                        # sigs2_raw
    ]).clone().detach().requires_grad_(True)

    # Compute J and solve for viable parameters
    j_score, viable_lam1, viable_lam2 = calculate_J_with_viable_lambdas(R1, R2, S0, theta)

    # Run backward pass to confirm gradients track through the linear scaling system
    j_score.backward()

    print("=== Viable Baseline Simulation ===")
    print(f"Calculated Viable Lambda 1 (rho): {viable_lam1.item():.6f}")
    print(f"Calculated Viable Lambda 2 (1-rho): {viable_lam2.item():.6f}")
    print(f"Sum of Lambdas: {(viable_lam1 + viable_lam2).item():.1f}")
    print(f"Smoothness Penalty J: {j_score.item():.8f}")
    print(f"Gradient Tensor Shape: {theta.grad.shape[0]}")
    print("\nGradients (dJ/dtheta) successfully computed:")
    print(theta.grad)

=== Viable Baseline Simulation ===
Calculated Viable Lambda 1 (rho): -162.346834
Calculated Viable Lambda 2 (1-rho): -1.262160
Sum of Lambdas: -163.6
Smoothness Penalty J: 0.00192326
Gradient Tensor Shape: 14

Gradients (dJ/dtheta) successfully computed:
tensor([-7.6928e-06, -3.4213e-06,  4.6835e-05, -5.7636e-05, -9.2803e-06,
        -8.4506e-05, -7.1026e-06,  4.5359e-05,  1.9105e-06,  3.1896e-09,
         3.1591e-09,  2.0298e-05, -1.3392e-06,  1.4864e-08])
